# E6 Exercises: Preprocessing Pipelines

> 2,400 outpatient referrals, mixed types · ~90 minutes · needs `notebooks/C6_Preprocessing_Pipelines.ipynb`

## Objective → task map

Every objective the chapter states is assessed by at least one task below. Where the
chapter's own embedded exercises already cover one, that is said here rather than
duplicated.

| # | Objective | Assessed by |
|---|---|---|
| 1 | Build a Pipeline of transformers ending in an estimator | Task 1 |
| 2 | Route column types separately with ColumnTransformer | Tasks 2, 5 |
| 3 | Cross-validate and grid-search a pipeline without leaking | Tasks 3, 4 |
| 4 | Reach a fitted step's parameters through named_steps | Tasks 1, 5 |
| 5 | Write a custom transformer with fit and transform | Task 5 |
| 6 | Save and reload a fitted pipeline | Task 5 Part 3 |

## Before you start

**Everything here is solvable with C6 alone.** `Pipeline`, `ColumnTransformer`, `SimpleImputer`, `StandardScaler`, `OneHotEncoder`, `GridSearchCV`, `cross_val_score`, `named_steps`, `joblib.dump` / `joblib.load`, and `BaseEstimator` + `TransformerMixin` for the custom transformer in Task 5.

Nothing below needs a method the chapter did not cover. Where a task is hard, it is
hard because of a decision you have to make and justify, not because of syntax.

**Marks.** 100 in total. Task 5 carries 45 and is an audit: four defects, one rebuild, and a question about which defect matters most.

**Dataset path used in this notebook:** `../data/clinic_referrals.csv`

---

## Task 1: Three steps, one object

**Warm-up · 10 marks · Build a Pipeline of transformers ending in an estimator**

Build a `Pipeline` with three steps: median imputation, standard scaling, and a logistic
regression. Fit it on the numeric columns `patient_age` and `wait_days` and report the
test ROC-AUC.

Then, without refitting anything, reach into the fitted pipeline and print:

- the median the imputer learned for `wait_days`
- the mean and scale the scaler learned for `patient_age`

Say which attribute of the pipeline you used to get at them.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df = pd.read_csv('../../00-data/clinic_referrals.csv', parse_dates=['referral_date'])
y = df['accepted']
NUM = ['patient_age', 'wait_days']
CAT = ['specialty', 'triage_band']

# Your code here

**Your answer to Task 1:**

*(write here)*

---

## Task 2: Different columns, different treatment

**Warm-up · 15 marks · Route numeric and categorical columns with ColumnTransformer**

The numeric columns need imputing and scaling. `specialty` and `triage_band` need
one-hot encoding and must **not** be scaled.

Build a `ColumnTransformer` that routes each group correctly, put it in front of a logistic
regression, and report the test AUC.

Then report the number of columns the transformer produces, and print the output feature
names.

Finally answer: **why should the one-hot columns not be scaled?** Note that scaling them
does not raise an error and barely changes the score, so the answer is not "it breaks".

In [ ]:
# Your code here

**Your answer to Task 2:**

*(write here)*

---

## Task 3: The same preprocessing, done by hand

**Core · 15 marks · Cross-validate a pipeline without leaking**

Cross-validate the Task 2 pipeline with `StratifiedKFold(5, shuffle=True,
random_state=42)` and `scoring='roc_auc'`. Report the mean.

Now do the same preprocessing by hand: impute, scale and one-hot the **whole dataset
first**, then cross-validate a bare logistic regression on the result. Report that mean.

Report both, and the difference. Then answer: **the two numbers are close. Does that mean
the manual version is acceptable?**

In [ ]:
# Your code here

**Your answer to Task 3:**

*(write here)*

---

## Task 4: Tuning the preprocessing, not just the model

**Core · 15 marks · Grid-search a pipeline including a preprocessing parameter**

Run a `GridSearchCV` over the Task 2 pipeline searching **both**:

- `pre__num__impute__strategy` in `['mean', 'median', 'most_frequent']`
- `clf__C` in `[0.01, 0.1, 1, 10]`

Report the best parameters, the best cross-validated score, and the full results table
sorted by rank.

Then answer: **explain the `pre__num__impute__strategy` name, one `__` at a time.** And:
**why can the imputation strategy only be tuned honestly inside a pipeline?**

In [ ]:
# Your code here

**Your answer to Task 4:**

*(write here)*

---

## Task 5: The audit

**Judgement · 45 marks · Find every leak in a hand-written workflow and rebuild it**

A colleague sends this notebook and asks you to sanity-check it before they present it.
It runs. It reports a cross-validated AUC of about 0.85.

```python
df = pd.read_csv('../../00-data/clinic_referrals.csv')
y = df['accepted']

# handle missing values
df['wait_days'] = df['wait_days'].fillna(df['wait_days'].median())

# scale the numeric columns
sc = StandardScaler()
df[['patient_age', 'wait_days']] = sc.fit_transform(df[['patient_age', 'wait_days']])

# encode the practice with target encoding, it has too many levels for one-hot
practice_means = df.groupby('gp_practice')['accepted'].mean()
df['gp_te'] = df['gp_practice'].map(practice_means)

# one-hot the small categoricals
df = pd.get_dummies(df, columns=['specialty', 'triage_band'])

X = df.drop(columns=['accepted', 'referral_id', 'gp_practice',
                     'referral_note', 'referral_date'])
score = cross_val_score(LogisticRegression(max_iter=2000), X, y, cv=5,
                        scoring='roc_auc').mean()
print(score)   # 0.85
```

**Part 1 (15 marks).** Identify every defect. There are four. For each one, state what it
is, and whether it inflates the reported score, makes the model undeployable, or both.

**Part 2 (15 marks).** Rebuild it as a single `Pipeline` with a `ColumnTransformer`,
cross-validate it honestly, and report the corrected score. Quantify how much of the
colleague's 0.85 was real.

**Part 3 (5 marks).** Fit your pipeline, save it with `joblib.dump`, load it back, and
show it predicts on a raw, untouched row of the original CSV. State what is inside the
saved file besides the model coefficients.

**Part 4 (10 marks).** One of the four defects would still be a defect even if it did not
change the score at all. Say which, and why that makes it the most serious of the four.

In [ ]:
# Your code here

**Your answer to Task 5:**

*(write here)*

---

## Before you submit

- [ ] Every transformer that learns anything is inside the Pipeline
- [ ] `OneHotEncoder` uses `handle_unknown='ignore'`
- [ ] Imputation comes before scaling in every numeric branch
- [ ] Your Task 5 pipeline predicts on a raw row straight from the CSV
- [ ] You have a `.joblib` file, and you can say what is inside it